In [1]:
import os, pickle, re
import numpy as np
import pandas as pd
import json

from models import carl, taylr
from physics.simulation import mcfm, msq
from physics.hstar import c6, eft
from physics.variables import cH_vals

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [2]:
torch.set_float32_matmul_precision('high')

import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": r"\usepackage{amssymb}",
})

In [3]:
run_dir = 'run/'

In [4]:
coeffs = [1, 2, 3, 4]
n_coeffs = len(coeffs)

def get_taylr_results(output_dir, coeffs):

    """
    Load the results from the taylr runs.
    Args:
        output_dir (str): Directory where the taylr runs are stored.
        coeffs (list): List of coefficients used in the taylr runs.
    Returns:
        events_test (list): List of test events.
        scaler_x (object): Scaler for the input features.
        models (list): List of loaded TAYLR models.
        scalers_y (list): List of scalers for the output features.
    """

    # testing events and feature scaler are the same for all runs
    first_dir = os.path.join(output_dir, f'taylr_{coeffs[0]}')
    with open(os.path.join(first_dir, 'events_test.pkl'), 'rb') as f:
        events_test = pickle.load(f)
    with open(os.path.join(first_dir, 'scaler_X.pkl'), 'rb') as f:
        scaler_x = pickle.load(f)

    models = []
    scalers_y = []

    ckpt_pattern = re.compile(r'epoch=(\d+)-val_loss=.*\.ckpt')

    for c in coeffs:
        taylr_dir = os.path.join(output_dir, f'taylr_{c}')
        
        # Load scaler_y
        with open(os.path.join(taylr_dir, 'scaler_y.pkl'), 'rb') as f:
            scaler_y = pickle.load(f)
        scalers_y.append(scaler_y)

        # Find latest version directory
        logs_dir = os.path.join(taylr_dir, 'lightning_logs')
        versions = [d for d in os.listdir(logs_dir) if d.startswith('version_') and d.split('_')[-1].isdigit()]
        if not versions:
            raise FileNotFoundError(f"No version directories found in {logs_dir}")
        latest_version = max(versions, key=lambda v: int(v.split('_')[-1]))

        # Find latest checkpoint
        checkpoint_dir = os.path.join(logs_dir, latest_version, 'checkpoints')
        all_ckpts = [f for f in os.listdir(checkpoint_dir) if ckpt_pattern.match(f)]
        if not all_ckpts:
            raise FileNotFoundError(f"No matching checkpoint files found in {checkpoint_dir}")
        all_ckpts.sort(key=lambda f: int(ckpt_pattern.match(f).group(1)), reverse=True)
        latest_ckpt = os.path.join(checkpoint_dir, all_ckpts[0])

        # Load model
        model = taylr.TAYLR.load_from_checkpoint(checkpoint_path=latest_ckpt)
        models.append(model)

    return events_test, scaler_x, models, scalers_y

events_gg_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l = get_taylr_results(os.path.join(run_dir,'h4l'), coeffs)
events_gg_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v = get_taylr_results(os.path.join(run_dir,'h2l2v'), coeffs)

In [5]:
def get_carl_results(output_dir):

    carl_dir = os.path.join(output_dir, 'carl')
    logs_dir = os.path.join(carl_dir, 'lightning_logs')

    with open(os.path.join(carl_dir, 'events_numerator_test.pkl'), 'rb') as f:
        events_test = pickle.load(f)
    with open(os.path.join(carl_dir, 'scaler.pkl'), 'rb') as f:
        scaler_x = pickle.load(f)

    # Find the latest version folder
    versions = [d for d in os.listdir(logs_dir) if re.match(r'version_\d+', d)]
    if not versions:
        raise FileNotFoundError("No version folders found in lightning_logs.")

    # Extract version numbers and sort
    latest_version = max(versions, key=lambda v: int(re.search(r'\d+', v).group()))
    checkpoint_dir = os.path.join(logs_dir, latest_version, 'checkpoints')

    # Find all checkpoint files matching the pattern
    checkpoints = [f for f in os.listdir(checkpoint_dir) if re.match(r'epoch=\d+-val_loss=.+\.ckpt', f)]
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")

    # Get the checkpoint with the largest epoch number
    ckpt_path = max(checkpoints, key=lambda f: int(re.search(r'epoch=(\d+)', f).group(1)))

    model = carl.CARL.load_from_checkpoint(checkpoint_path=os.path.join(checkpoint_dir, ckpt_path))

    return events_test, scaler_x, model

events_qq_4l, scaler_carl_4l, model_carl_4l = get_carl_results(os.path.join(run_dir,'h4l'))
events_qq_2l2v, scaler_carl_2l2v, model_carl_2l2v = get_carl_results(os.path.join(run_dir,'h2l2v'))

In [6]:
c6_linspace = [-20,20,201]
cHbox_linspace = [-0.02,0.05,141]

c6_space = np.linspace(*c6_linspace)
cHbox_space = np.linspace(*cHbox_linspace)

c6_sm = 0.0
cHbox_sm = 0.0

i_c6_sm = np.where(c6_space==c6_sm)[0][0]
i_cHbox_sm = np.where(np.round(cHbox_space,5)==cHbox_sm)[0][0]

c6_asimov = 0.0
cHbox_asimov = 0.0

i_c6_asimov = np.where(c6_space==c6_asimov)[0][0]
i_cHbox_asimov = np.where(np.round(cHbox_space,5)==cHbox_asimov)[0][0]

In [8]:
lumi = 3000

with open('data/zz4l/xsec.json', 'r') as f:
    xs_4l = json.load(f)
with open('data/zz2l2v/xsec.json', 'r') as f:
    xs_2l2v = json.load(f)

events_gg_4l.weights *= np.prod(xs_4l['gg_sbi']) / events_gg_4l.weights.sum()
events_gg_2l2v.weights *= np.prod(xs_2l2v['gg_sbi']) / events_gg_2l2v.weights.sum() * 0.15

events_qq_4l.weights *= np.prod(xs_4l['qq']) / events_qq_4l.weights.sum()
events_qq_2l2v.weights *= np.prod(xs_2l2v['qq']) / events_qq_2l2v.weights.sum() * 0.15

In [9]:
sample_size = 50_000

class ZeroWeightFilter():
    def __init__(self):
        pass
    def __call__(self, kinematics, components, weights, probabilities):
        return np.where(weights != 0)

events_gg_4l, events_gg_2l2v = events_gg_4l.sample(sample_size), events_gg_2l2v.filter(ZeroWeightFilter()).sample(sample_size)
events_qq_4l, events_qq_2l2v = events_qq_4l.sample(sample_size), events_qq_2l2v.filter(ZeroWeightFilter()).sample(sample_size)

w_gg_4l_sm, w_gg_2l2v_sm = events_gg_4l.weights.to_numpy(), events_gg_2l2v.weights.to_numpy()
w_qq_4l_sm, w_qq_2l2v_sm = events_qq_4l.weights.to_numpy(), events_qq_2l2v.weights.to_numpy()

sigma_gg_4l_sm = np.sum(w_gg_4l_sm)
sigma_gg_2l2v_sm = np.sum(w_gg_2l2v_sm)
sigma_qq_4l_sm = np.sum(w_qq_4l_sm)
sigma_qq_2l2v_sm = np.sum(w_qq_2l2v_sm)

events_4l = mcfm.stack(events_gg_4l, events_qq_4l)
events_2l2v = mcfm.stack(events_gg_2l2v, events_qq_2l2v)
sigma_4l_sm = events_4l.weights.sum()
sigma_2l2v_sm = events_2l2v.weights.sum()

events_gg = mcfm.stack(events_gg_4l, events_gg_2l2v)
events_qq = mcfm.stack(events_qq_4l, events_qq_2l2v)
sigma_gg_sm = events_gg.weights.sum()
sigma_qq = events_qq.weights.sum()

events_pp = mcfm.stack(events_gg, events_qq)
sigma_pp_sm = events_pp.weights.sum()

In [10]:
c6_mod = c6.Modifier(baseline=msq.Component.SBI, events=events_gg, c6_points=[-20, -10, 0, 10, 20])
w_gg_c6_true, p_c6_true = c6_mod.modify(c6_values = c6_space)
w_gg_bsm_true = w_gg_c6_true[:,i_c6_asimov] - 2*cHbox_asimov*w_gg_c6_true[:,i_c6_sm]
sigma_gg_bsm_true = np.sum(w_gg_bsm_true, axis=0)  # c6 & cHbox

In [11]:
features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
                'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
                'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
                'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
features_2l2v = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

batch_size = 1024

In [12]:
trainer = L.Trainer(accelerator='gpu', devices=1)

kinematics_4l = events_4l.kinematics[features_4l]
kinematics_2l2v = events_2l2v.kinematics[features_2l2v]

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


In [13]:

X_carl_4l = scaler_carl_4l.transform(kinematics_4l.to_numpy())
dl_carl = DataLoader(TensorDataset(torch.tensor(X_carl_4l, dtype=torch.float32)), batch_size=batch_size, num_workers=8)
s_carl_4l = torch.concatenate(trainer.predict(model_carl_4l, dl_carl)).numpy()

X_carl_2l2v = scaler_carl_2l2v.transform(kinematics_2l2v.to_numpy())
dl_carl = DataLoader(TensorDataset(torch.tensor(X_carl_2l2v, dtype=torch.float32)), batch_size=batch_size, num_workers=8) 
s_carl_2l2v = torch.concatenate(trainer.predict(model_carl_2l2v, dl_carl)).numpy()

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [14]:
r_4l = s_carl_4l/(1-s_carl_4l)
r_2l2v = s_carl_2l2v/(1-s_carl_2l2v)
r = np.concatenate((r_4l, r_2l2v))

In [15]:
X_4l = scaler_X_taylr_4l.transform(events_4l.kinematics[features_4l].to_numpy())
X_4l = torch.tensor(X_4l, dtype=torch.float32) 
dl_4l = DataLoader(TensorDataset(X_4l), batch_size=batch_size, num_workers=8)

coeffs_4l = []
for i in range(n_coeffs):
    coeffs_4l.append(scalers_y_taylr_4l[i].inverse_transform(torch.concatenate(trainer.predict(models_taylr_4l[i], dl_4l)).numpy()[:,np.newaxis]).flatten())
coeffs_4l = np.array(coeffs_4l).T

X_2l2v = scaler_X_taylr_2l2v.transform(events_2l2v.kinematics[features_2l2v].to_numpy())
X_2l2v = torch.tensor(X_2l2v, dtype=torch.float32) 
dl_2l2v = DataLoader(TensorDataset(X_2l2v), batch_size=batch_size, num_workers=8)

coeffs_2l2v = []
for i in range(n_coeffs):
    coeffs_2l2v.append(scalers_y_taylr_2l2v[i].inverse_transform(torch.concatenate(trainer.predict(models_taylr_2l2v[i], dl_2l2v)).numpy()[:,np.newaxis]).flatten())
coeffs_2l2v = np.array(coeffs_2l2v).T

coeffs = np.vstack((coeffs_4l, coeffs_2l2v))

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [16]:
def f(c6_points, coeffs):
    coefficients = np.concatenate([np.ones((len(coeffs),1)), coeffs], axis=1)
    c6_matrix = np.vander(c6_points, coefficients.shape[1], increasing=True).T
    return np.dot(coefficients, c6_matrix)

In [17]:
X_gg_4l = scaler_X_taylr_4l.transform(events_gg_4l.kinematics[features_4l].to_numpy())
X_gg_4l = torch.tensor(X_gg_4l, dtype=torch.float32) 
dl_gg_4l = DataLoader(TensorDataset(X_gg_4l), batch_size=batch_size, num_workers=8)

coeffs_gg_4l = []
for i in range(n_coeffs):
    coeffs_gg_4l.append(scalers_y_taylr_4l[i].inverse_transform(torch.concatenate(trainer.predict(models_taylr_4l[i], dl_gg_4l)).numpy()[:,np.newaxis]).flatten())
coeffs_gg_4l = np.array(coeffs_gg_4l).T

X_gg_2l2v = scaler_X_taylr_2l2v.transform(events_gg_2l2v.kinematics[features_2l2v].to_numpy())
X_gg_2l2v = torch.tensor(X_gg_2l2v, dtype=torch.float32) 
dl_gg_2l2v = DataLoader(TensorDataset(X_gg_2l2v), batch_size=batch_size, num_workers=8)

coeffs_gg_2l2v = []
for i in range(n_coeffs):
    coeffs_gg_2l2v.append(scalers_y_taylr_2l2v[i].inverse_transform(torch.concatenate(trainer.predict(models_taylr_2l2v[i], dl_gg_2l2v)).numpy()[:,np.newaxis]).flatten())
coeffs_gg_2l2v = np.array(coeffs_gg_2l2v).T

coeffs_gg = np.vstack((coeffs_gg_4l, coeffs_gg_2l2v))

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [18]:
# get ct parameterization
eft_mod = eft.Modifier(events=events_gg_4l, baseline=msq.Component.SBI)


In [19]:
print(eft_mod.coefficients[321][0][1][0])
print(eft_mod.coefficients[123][0][1][0])
print(eft_mod.coefficients[1234][0][1][0])

-0.032353626584214534
-0.051203627527034556
-0.06219827262118352


In [20]:
w_gg_sm = events_gg.weights.to_numpy()
w_gg_c6 = w_gg_sm[:,np.newaxis] * f(c6_space, coeffs_gg)
w_gg_bsm = w_gg_c6[:,:,np.newaxis] - 2 * (w_gg_sm[:,np.newaxis,np.newaxis] * cHbox_space[np.newaxis,np.newaxis,:])

sigma_gg_sm = np.sum(w_gg_sm)
sigma_gg_c6 = np.sum(w_gg_c6, axis=0)  # c6 & cHbox
sigma_gg_bsm = np.sum(w_gg_bsm, axis=0)  # c6 & cHbox

In [21]:
# TEST: this should mess with everything below
# w_pp_sm = mcfm.stack(events_gg_2l2v,events_qq_4l,events_gg_4l,events_qq_2l2v).weights.to_numpy()

w_pp_sm = events_pp.weights.to_numpy()
p_pp_sm = events_pp.weights.to_numpy() / sigma_pp_sm

sigma_pp_sm = sigma_gg_sm + sigma_qq
sigma_pp_bsm = sigma_gg_bsm + sigma_qq 

nu_pp_sm = sigma_pp_sm * lumi
nu_pp_bsm = sigma_pp_bsm * lumi

In [22]:
n_gg_asimov = sigma_gg_sm*lumi
n_pp_asimov = sigma_pp_sm*lumi

# Poisson term
t_rate = -2 * nu_pp_sm * (np.log(nu_pp_bsm) - np.log(nu_pp_sm)) + 2 * (nu_pp_bsm - nu_pp_sm) 

$$
\frac{p (x|\theta)}{p (x|0)} = \frac{\sigma_{gg}(0) + \sigma_{qq}}{\sigma_{gg}(\theta) + \sigma_{qq}} \frac{ \sigma_{gg}(0)\sum a_k(x) \theta^k + \sigma_{qq}\frac{s(x)}{1-s(x)} }{ \sigma_{gg}(0) + \sigma_{qq}\frac{s(x)}{1-s(x)} }
$$

In [23]:
taylr_gg_c6 = sigma_gg_sm * f(c6_space, coeffs)
carl_qq = sigma_qq * r[:,np.newaxis]
pratio_c6 =  (sigma_gg_sm + sigma_qq)/(sigma_gg_c6+sigma_qq) * (taylr_gg_c6 + carl_qq)/(sigma_gg_sm + carl_qq)

In [24]:
p_pp_c6 = pratio_c6 * p_pp_sm[:,np.newaxis]

In [25]:
psum_pp_c6 = np.sum(p_pp_c6, axis=0)  # c6 & cHbox

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

ax.plot(c6_space, psum_pp_c6)

plt.savefig('pratio_calibration.pdf', bbox_inches='tight')

In [26]:
pratio_c6_calibrated = pratio_c6 / psum_pp_c6

In [27]:
t_shape = - 2 * np.sum(lumi * w_pp_sm[:,np.newaxis,np.newaxis] * np.log(pratio_c6_calibrated)[:,:,np.newaxis] , axis=0)

In [28]:
t = t_rate + t_shape 
# t = t_rate

In [29]:
i_c6_fit = np.nanargmin(t) // cHbox_linspace[2]
i_cHbox_fit = np.nanargmin(t)  % cHbox_linspace[2]

i_c6_fit_rate = np.nanargmin(t_rate) // cHbox_linspace[2]
i_cHbox_fit_rate = np.nanargmin(t_rate)  % cHbox_linspace[2]

t_min = t[i_c6_fit,i_cHbox_fit]
t_min_rate = t_rate[i_c6_fit_rate,i_cHbox_fit_rate]

In [30]:
X, Y = np.meshgrid(c6_space, cHbox_space)
Z = t.T
Z_rate = t_rate.T

In [31]:
plt.figure(figsize=(6,4))

contours = plt.contour(X, Y, Z, levels=[t_min+1,t_min+4], colors='royalblue', linestyles=['--','-'])
contours_rate = plt.contour(X, Y, Z_rate, levels=[t_min_rate+1,t_min_rate+4], colors='grey', linestyles=['--','-'])

plt.scatter(c6_space[i_c6_fit], cHbox_space[i_cHbox_fit], marker='+', color='blue')

labels = [
    Line2D([0],[0],color='grey',label='$\\mathrm{Rate}$'),
    Line2D([0],[0],color='black',label='$\\mathrm{NSBI}$'),
]
plt.legend(frameon=False, handles=labels, loc='upper center')

plt.xlabel('$C_H$', fontsize=12)
plt.ylabel('$C_{H\\square}$', fontsize=12)

plt.xlim(-20,20)
plt.ylim(-0.015,0.03)

plt.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig('nll_pp.pdf', bbox_inches='tight')